## The following Python notebook represents code that would be called from a Notebook Server webhook receiver, triggered by an incoming webhook.


#### Run this cell to connect to your GIS and get started:

In [1]:
from arcgis.gis import GIS
gis = GIS("home")

You are logged on as admin with an administrator role, proceed with caution.


#### Functions to get the features and attributes from the /ExtractChanges API within the Feature Service

In [ ]:
# When developing your notebook, use an example payload, one that would be similiar to a message that triggers the notebook
samplePayload = {"serviceType":"FeatureServer","changesUrl":"https://example.com/server/rest/services/Hosted/MyService/FeatureServer/extractChanges?serverGens=%5B1773230457695,1773244883393%5D","name":"Posting","id":"efab8ba4-6a0b-491e-8624-73a25e15ffb0","folderName":"","serviceName":"MyService","events":[{"eventType":"FeaturesUpdated","when":1741189389713}]}
# After putting your notebook into production, this cell can be deleted

In [ ]:
# Functions to get the features and attributes from the /ExtractChanges API within the Feature Service

from urllib.parse import urlparse
from urllib.parse import parse_qs
from urllib.parse import unquote
import time

def parseResults(resultJSONURL, params=None):

    resultJSON = gis._con.post(resultJSONURL, params, files=None)
    if "edits" in resultJSON:
        print("Found  {}  layers with edits in the changes file".format(len(resultJSON['edits'])))
    else:
        print("No edits found")

    return resultJSON


def waitForStauts(statusURL, params=None):

    statusPayload = {"status":"foo"}
    counter = 0    

    while statusPayload["status"].upper() != "COMPLETED":
        statusPayload = gis._con.post(statusURL, params)
        counter += 1
        time.sleep(1)
        if counter == 20:
            print("No results after 20 seconds, something is probably wrong")
            continue
    
    try:
        return statusPayload['resultUrl']
    except Exception:
        print("Failed to get a resultURL")


def getExtractChanges(changesURL):
    
    # Depending on the type of changes happening in your feature service,
    #  the following parameters can be updated to get back specific types
    #  of changes. Below is set to return Inserts (add) only.

    parse = urlparse(changesURL)
    extract_params = {
        "serverGens": parse_qs(parse.query)['serverGens'][0],
        "returnInserts": "true",
        "returnUpdates": "false",
        "returnDeletes": "false",
        "returnAttachments": "false",
        "returnAttachmentsDataByUrl": "false"
    }

    changePayload = gis._con.post(changesURL.split("?")[0], extract_params)

    if "statusUrl" in changePayload:
        return changePayload['statusUrl']
    else:
        print("No status URL found, cannot proceed")

def doExtractChanges(changesURL):

    statusURL = getExtractChanges(changesURL)

    resultJSONURL = waitForStauts(statusURL)

    changeResults = parseResults(resultJSONURL)

    return changeResults

### Get the changeURL from the payload and use that to get the change JSON via Extract Change

In [ ]:
# Get the changeURL from the payload and use that to get the change JSON via Extract Changes
import json

changeInfo = doExtractChanges(samplePayload['changesUrl'])

# Comment out the above and uncomment this line to use the code in production, where the payload is coming from the 
#   trigger message instead of a sample payload
# 12.1+ example
#changeInfo = doExtractChanges(webhookPayload['changesUrl'])
# 11.5-12.0 example
#changeInfo = doExtractChanges(json.loads(webhookPayload['changesUrl']))

print(changeInfo)

### Look through the edits to determine if further action is required

In [6]:
updates = []
for e in changeInfo['edits']:
    if 'updates' in e['features']:
        updates = e['features']['updates']
        break

if not updates:
    print("No new updates")
else:
    print(updates)

[{'attributes': {'rankcondition': 2, 'globalid': '{4F95E7F8-8065-419C-AB38-C2E1E37A70BF}', 'reviewed': 'Yes', 'collectorid': 2, 'objectid': 55}, 'geometry': {'x': 368120.28500000015, 'y': 5031795.9815, 'z': 0}}, {'attributes': {'rankcondition': 3, 'globalid': '{32CE8264-899B-4CB5-BFCE-3A73C351CDF2}', 'reviewed': 'Yes', 'collectorid': 2, 'objectid': 78}, 'geometry': {'x': 368093.27809999976, 'y': 5031804.243799999, 'z': 0}}, {'attributes': {'rankcondition': 3, 'globalid': '{7D86B33B-78C2-48A0-8686-D420780CA8EA}', 'reviewed': 'Yes', 'collectorid': 2, 'objectid': 85}, 'geometry': {'x': 368086.0548999999, 'y': 5031809.3959, 'z': 0}}]


### Do something with the edits....

In [ ]:
if updates:    
    # Iterate through each new posted feature    
    for a in updates:        
        for u in updates:
            # Eg. If attribute "reviewed" == yes, do something...
            if u['attributes']['reviewed'] == 'Yes':
                # do something...
                pass